In [1]:
import pandas as pd
import importlib

from gptnl_dataloader.sampling import functions

# Load curated dataset level descriptor

In [2]:
mapping_column_names = {
    "en": "Tokens (B) English",
    "nl": "Tokens (B) Dutch",
    "code": "Est. Tokens (B)",  # is_code is a separate column, used as filter
    "other": "Tokens (B) Other languages",
}
mapping_column_names_nocode = {
    k: v for k, v in mapping_column_names.items() if k != "code"
}

# To re-run and update this one, run explore-curated-data.ipynb
metrics_df = pd.read_csv("../curated_data_overview.csv")
metrics_df

## Set target distribution

In [3]:
# Constants for sampling
VALIDATION_RATIO = 0.05
ANNEALING_RATIO = 0.15  # of remaining data after validation

PRIMARY_LANGUAGE_DIST = {"en": 0.50, "nl": 0.25, "code": 0.15, "other": 0.10}

ANNEALING_LANGUAGE_DIST = {"en": 0.45, "nl": 0.35, "code": 0.10, "other": 0.10}

In [4]:
importlib.reload(functions)

In [5]:
primary_available_df = metrics_df
# primary: upsampling
primary_available = functions.get_actual_counts(
    primary_available_df, mapping_column_names
)
print(primary_available)
mul_factors = functions.calculate_scaling_factors(
    primary_available, PRIMARY_LANGUAGE_DIST
)
# if target is 33:33:33 (1:1:1) and real is 25:25:50 (1:1:2), scaling should be (2:2:1)
print(mul_factors)

{'en': 224.63, 'nl': 17.27, 'code': 30.93, 'other': 33.61}
{'en': 1.0, 'nl': 6.503474232773597, 'code': 2.1787584869059167, 'other': 1.3366855102648023}


In [6]:
annealing_available_df = metrics_df[
    (metrics_df["Quality"] == "high") | metrics_df["is_code"]
]
annealing_available = functions.get_actual_counts(
    annealing_available_df, mapping_column_names
)
print(annealing_available)
# annealing: upsampling
mul_factors = functions.calculate_scaling_factors(
    annealing_available, ANNEALING_LANGUAGE_DIST
)
print(mul_factors)
# annealing: downsampling
mul_factors = functions.calculate_scaling_factors(
    annealing_available, PRIMARY_LANGUAGE_DIST, upsample=False
)
print(mul_factors)

{'en': 65.17, 'nl': 11.910000000000002, 'code': 30.93, 'other': 9.42}
{'en': 2.135721957956115, 'nl': 9.089420654911837, 'code': 1.0, 'other': 3.2834394904458604}
{'en': 0.36550560073653526, 'nl': 1.0, 'code': 0.23103782735208536, 'other': 0.5057324840764332}


In [7]:
# How to actually get desired distribution (with approx factor) without doing row-wise sampling but only file-wise sampling? (e.g. need 1.41 scaling factor)
# In other words, repeat single files discrete number of times (not 1.5, either 1 or 2)
# 1. get smaller partial files so that we can randomly select some files to repeat (e.g., scaling needed 1.41, repeat 100 files 1x and 41 files 2x)
# 2. multiply every scaling factor xN: e.g., scaling needed 1.41: from (1, 1.41) to (5, 7.05) and then approximate (in this case 0.7%)
#    a. approximation limit (want to be enough similar to target)
#    b. cannot do very big numbers (maximum is in the tens?)

### File-based

In [8]:
# general file stats
all_files_paths_by_bucket = functions.find_bucket_files()
all_files_size_by_bucket, all_files_size_by_path = functions.analyse_sizes(
    all_files_paths_by_bucket
)

# split
train_files_paths_by_bucket, validate_files_paths_by_bucket = (
    functions.split_train_validation(
        files_paths_by_bucket=all_files_paths_by_bucket,
        files_size_by_path=all_files_size_by_path,
    )
)
# train
train_files_size_by_bucket, train_files_size_by_path = functions.analyse_sizes(
    train_files_paths_by_bucket
)
# validate
validate_files_size_by_bucket, validate_files_size_by_path = functions.analyse_sizes(
    validate_files_paths_by_bucket
)


for bucket in functions.ALL_BUCKETS_DEFAULT:
    print(bucket)
    print(
        f"     ALL      {len(all_files_paths_by_bucket[bucket]): >5} files, total {all_files_size_by_bucket[bucket] >> 30: >5} GB"
    )
    print(
        f"     TRAIN    {len(train_files_paths_by_bucket[bucket]): >5} files, total {train_files_size_by_bucket[bucket] >> 30: >5} GB"
    )
    print(
        f"     VALIDATE {len(validate_files_paths_by_bucket[bucket]): >5} files, total {validate_files_size_by_bucket[bucket] >> 30: >5} GB"
    )

english_high_quality
     ALL         99 files, total    92 GB
     TRAIN       96 files, total    90 GB
     VALIDATE     3 files, total     2 GB
english_low_medium_quality
     ALL        489 files, total   419 GB
     TRAIN      466 files, total   399 GB
     VALIDATE    23 files, total    19 GB
dutch_high_quality
     ALL         32 files, total    29 GB
     TRAIN       32 files, total    29 GB
     VALIDATE     0 files, total     0 GB
dutch_low_medium_quality
     ALL         16 files, total    14 GB
     TRAIN       16 files, total    14 GB
     VALIDATE     0 files, total     0 GB
code
     ALL        152 files, total   146 GB
     TRAIN      146 files, total   141 GB
     VALIDATE     6 files, total     5 GB
other_languages
     ALL        107 files, total    98 GB
     TRAIN      103 files, total    94 GB
     VALIDATE     4 files, total     3 GB


In [9]:
# primary
primary_available_file_based = functions.aggregate_groups(
    train_files_size_by_bucket, functions.PRIMARY_GROUPS_DEFAULT
)

print({k: f"{v >> 30} GB" for k, v in primary_available_file_based.items()})

mul_factors_primary_file_based = functions.calculate_scaling_factors(
    primary_available_file_based, PRIMARY_LANGUAGE_DIST
)
print("FILE-based mul_factors", mul_factors_primary_file_based)
# they are quite similar (not anymore, csv is not updated)

{'en': '490 GB', 'nl': '44 GB', 'code': '141 GB', 'other': '94 GB'}
FILE-based mul_factors {'en': 1.0, 'nl': 5.566133887614951, 'code': 1.0418007785552383, 'other': 1.034598003809675}


In [10]:
# annealing
annealing_available_file_based = functions.aggregate_groups(
    train_files_size_by_bucket, functions.ANNEALING_GROUPS_DEFAULT
)

print({k: f"{int(v) >> 30} GB" for k, v in annealing_available_file_based.items()})

mul_factors_annealing_file_based_down = functions.calculate_scaling_factors(
    annealing_available_file_based, ANNEALING_LANGUAGE_DIST, upsample=False
)
print("FILE-based mul_factors downsampling", mul_factors_annealing_file_based_down)
# they are quite similar (not anymore)

{'en': '90 GB', 'nl': '29 GB', 'code': '141 GB', 'other': '94 GB'}
FILE-based mul_factors downsampling {'en': 0.4167718241588198, 'nl': 1.0, 'code': 0.05916372747675077, 'other': 0.08813202428722379}


In [11]:
# file-based oversampling

print("primary upscaling")
primary_oversampling_paths = functions.get_file_based_sampling(
    functions.PRIMARY_GROUPS_DEFAULT,
    mul_factors=mul_factors_primary_file_based,
    files_paths_by_bucket=train_files_paths_by_bucket,
    file_size_by_path=train_files_size_by_path,
    debug=True,
)
print("\nannealing downscaling")
annealing_oversampling_paths_down = functions.get_file_based_sampling(
    functions.ANNEALING_GROUPS_DEFAULT,
    mul_factors=mul_factors_annealing_file_based_down,
    files_paths_by_bucket=train_files_paths_by_bucket,
    file_size_by_path=train_files_size_by_path,
    debug=True,
)

primary upscaling
en:english_high_quality:
      mul_factor=  1.000 n_files=   96 full_repetitions:     1 partial repetition:     0 oversampling_paths:    96
      SIZE: total_input_size=92302 desired_bucket_size=92302 oversampled_size=92302
en:english_low_medium_quality:
      mul_factor=  1.000 n_files=  466 full_repetitions:     1 partial repetition:     0 oversampling_paths:   466
      SIZE: total_input_size=409471 desired_bucket_size=409471 oversampled_size=409471
nl:dutch_high_quality:
      n_files_partial_repetition(higher)=20 lower=16928MB higher=18025MB target=16938 MB
      mul_factor=  5.566 n_files=   32 full_repetitions:     5 partial repetition:    20 oversampling_paths:   180
      SIZE: total_input_size=29920 desired_bucket_size=166541 oversampled_size=168623
nl:dutch_low_medium_quality:
      n_files_partial_repetition(higher)=11 lower= 8091MB higher= 9378MB target=8578 MB
      mul_factor=  5.566 n_files=   16 full_repetitions:     5 partial repetition:    11 oversa

In [12]:
# verify actual distribution
print("primary")
primary_tot_size, primary_tot_size_by_group = functions.compare_file_based_sampling(
    PRIMARY_LANGUAGE_DIST,
    mul_factors_primary_file_based,
    primary_oversampling_paths,
    train_files_size_by_path,
    debug=True,
)

# print("\nannealing")
# functions.compare_file_based_sampling(ANNEALING_LANGUAGE_DIST, mul_factors_annealing_file_based, annealing_oversampling_paths, files_size_by_path)

print("\nannealing downsampling")
annealing_tot_size, annealing_tot_size_by_group = functions.compare_file_based_sampling(
    ANNEALING_LANGUAGE_DIST,
    mul_factors_annealing_file_based_down,
    annealing_oversampling_paths_down,
    train_files_size_by_path,
    debug=True,
)

primary
total size: 985 GB
   en: size_gb:   490GB 	 mul_factor:   1.000 	 target: 0.5 	 actual_dist: 0.4970
   nl: size_gb:   248GB 	 mul_factor:   5.566 	 target: 0.25 	 actual_dist: 0.2524
 code: size_gb:   148GB 	 mul_factor:   1.042 	 target: 0.15 	 actual_dist: 0.1509
other: size_gb:    98GB 	 mul_factor:   1.035 	 target: 0.1 	 actual_dist: 0.0996

annealing downsampling
total size: 85 GB
   en: size_gb:    37GB 	 mul_factor:   0.417 	 target: 0.45 	 actual_dist: 0.4404
   nl: size_gb:    29GB 	 mul_factor:   1.000 	 target: 0.35 	 actual_dist: 0.3424
 code: size_gb:     8GB 	 mul_factor:   0.059 	 target: 0.1 	 actual_dist: 0.1015
other: size_gb:     9GB 	 mul_factor:   0.088 	 target: 0.1 	 actual_dist: 0.1157


In [13]:
# want annealing to have a certain size
target_annealing_size = primary_tot_size * ANNEALING_RATIO
print("target_annealing_size", int(target_annealing_size) >> 30, "GB")
scaling_factor_global_annealing = target_annealing_size / annealing_tot_size
print("scaling_factor_global_annealing", scaling_factor_global_annealing)
mul_factors_annealing_file_based_scaled_to_ratio = {
    k: v * scaling_factor_global_annealing
    for k, v in mul_factors_annealing_file_based_down.items()
}
print(mul_factors_annealing_file_based_scaled_to_ratio)

target_annealing_size 147 GB
scaling_factor_global_annealing 1.732772327220134
{'en': 0.7221706836674587, 'nl': 1.732772327220134, 'code': 0.10251726974690722, 'other': 0.15271273282679415}


In [14]:
# final resized annealing with certain size
annealing_oversampling_paths_scaled = functions.get_file_based_sampling(
    functions.ANNEALING_GROUPS_DEFAULT,
    mul_factors=mul_factors_annealing_file_based_scaled_to_ratio,
    files_paths_by_bucket=train_files_paths_by_bucket,
    file_size_by_path=train_files_size_by_path,
    debug=True,
)
annealing_tot_size_scaled, annealing_tot_size_by_group_scaled = (
    functions.compare_file_based_sampling(
        ANNEALING_LANGUAGE_DIST,
        mul_factors_annealing_file_based_scaled_to_ratio,
        annealing_oversampling_paths_scaled,
        train_files_size_by_path,
        debug=True,
    )
)

en:english_high_quality:
      n_files_partial_repetition(higher)=71 lower=66174MB higher=67142MB target=66658 MB
      mul_factor=  0.722 n_files=   96 full_repetitions:     0 partial repetition:    71 oversampling_paths:    71
      SIZE: total_input_size=92302 desired_bucket_size=66658 oversampled_size=68109
nl:dutch_high_quality:
      n_files_partial_repetition(lower)=23 lower=21009MB higher=21986MB target=21924 MB
      mul_factor=  1.733 n_files=   32 full_repetitions:     1 partial repetition:    23 oversampling_paths:    55
      SIZE: total_input_size=29920 desired_bucket_size=51845 oversampled_size=51906
code:code:
      n_files_partial_repetition(lower)=15 lower=13828MB higher=14818MB target=14812 MB
      mul_factor=  0.103 n_files=  146 full_repetitions:     0 partial repetition:    15 oversampling_paths:    15
      SIZE: total_input_size=144492 desired_bucket_size=14812 oversampled_size=14818
other:other_languages:
      n_files_partial_repetition(lower)=16 lower=14077M